# Part E — Deep Learning Using PyTorch

Problems 16–18. This notebook uses Fashion-MNIST and a small feed-forward neural network. To keep CPU training practical, the experiments use fixed subsets of the dataset. Run all cells in order.

## Problem 16. Tensor operations

In [ ]:
import torch

a = torch.tensor([1, 2, 3], dtype=torch.float32)
b = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
c = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

print("Vector:", a, "shape:", a.shape, "dtype:", a.dtype)
print("Matrix shape:", b.shape)
print("Addition:", b + c)
print("Element-wise multiplication:", b * c)
print("Matrix multiplication:", b @ c)
print("Reshaped vector:", torch.arange(12).reshape(3, 4))

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 2 * x
y.backward()
print("Function value:", y.item())
print("Gradient dy/dx at x=3:", x.grad.item())

## Problem 17. Neural network implementation

In [ ]:
import time
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

transform = transforms.ToTensor()
train_full = datasets.FashionMNIST(root=".", train=True, download=True, transform=transform)
test_full = datasets.FashionMNIST(root=".", train=False, download=True, transform=transform)
train_set = Subset(train_full, range(0, 10000))
validation_set = Subset(train_full, range(10000, 12000))
test_set = Subset(test_full, range(0, 2000))
class_names = train_full.classes
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
validation_loader = DataLoader(validation_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)
print("Training samples:", len(train_set))
print("Validation samples:", len(validation_set))
print("Test samples:", len(test_set))
print("Class names:", class_names)

class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, images):
        return self.network(images)

def evaluate_model(model, loader, loss_function):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_function(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

def train_experiment(learning_rate, epochs=2):
    model = FashionClassifier().to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"train_loss": [], "train_accuracy": [], "val_loss": [], "val_accuracy": []}
    start = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_accuracy = correct / total
        val_loss, val_accuracy = evaluate_model(model, validation_loader, loss_function)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)
        print(f"Epoch {epoch + 1}/{epochs} | train loss {train_loss:.4f} | train accuracy {train_accuracy:.4f} | val loss {val_loss:.4f} | val accuracy {val_accuracy:.4f}")

    elapsed = time.perf_counter() - start
    test_loss, test_accuracy = evaluate_model(model, test_loader, loss_function)
    return model, history, elapsed, test_loss, test_accuracy

experiments = {}
for label, learning_rate in [("lr_0.001", 0.001), ("lr_0.0005", 0.0005)]:
    print("\nExperiment:", label)
    experiments[label] = train_experiment(learning_rate, epochs=2)

summary = []
for label, (model, history, elapsed, test_loss, test_accuracy) in experiments.items():
    summary.append({
        "Setting": label,
        "Learning rate": float(label.split("_")[1]),
        "Train accuracy": history["train_accuracy"][-1],
        "Validation accuracy": history["val_accuracy"][-1],
        "Test accuracy": test_accuracy,
        "Training time (seconds)": elapsed,
        "Test loss": test_loss
    })
experiment_table = pd.DataFrame(summary) if "pd" in globals() else __import__("pandas").DataFrame(summary)
display(experiment_table.round(4))
best_label = max(experiments, key=lambda key: experiments[key][1]["val_accuracy"])
best_model, best_history, best_time, best_test_loss, best_test_accuracy = experiments[best_label]
print("Selected setting using validation accuracy:", best_label)
torch.save(best_model.state_dict(), "pytorch_model.pth")
pytorch_results = {
    "framework": "PyTorch",
    "setting": best_label,
    "learning_rate": float(best_label.split("_")[1]),
    "epochs": 2,
    "train_accuracy": best_history["train_accuracy"][-1],
    "validation_accuracy": best_history["val_accuracy"][-1],
    "test_accuracy": best_test_accuracy,
    "training_time_seconds": best_time,
    "test_loss": best_test_loss,
    "parameter_count": sum(parameter.numel() for parameter in best_model.parameters()),
    "train_samples": len(train_set),
    "validation_samples": len(validation_set),
    "test_samples": len(test_set)
}
Path("pytorch_results.json").write_text(json.dumps(pytorch_results, indent=2))
print("Saved pytorch_model.pth and pytorch_results.json")

## Training plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for label, (_, history, _, _, _) in experiments.items():
    axes[0].plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], marker="o", label=label)
    axes[1].plot(range(1, len(history["val_accuracy"]) + 1), history["val_accuracy"], marker="o", label=label)
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].set_title("Validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
for axis in axes:
    axis.legend()
plt.tight_layout()
plt.savefig("pytorch_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Problem 18. Training experiments and saved-model prediction

In [ ]:
print("The two runs use the same architecture, data subsets, batch size and epoch count; only the learning rate changes.")
display(experiment_table.round(4))
print("The better setting is selected by validation accuracy, not test accuracy.")
loaded_model = FashionClassifier().to(device)
loaded_model.load_state_dict(torch.load("pytorch_model.pth", map_location=device, weights_only=True))
loaded_model.eval()
sample_images, sample_labels = next(iter(test_loader))
with torch.no_grad():
    sample_predictions = loaded_model(sample_images[:5].to(device)).argmax(dim=1).cpu().numpy()
for index, prediction in enumerate(sample_predictions):
    print(f"Sample {index + 1}: predicted {class_names[int(prediction)]}; actual {class_names[int(sample_labels[index])]}")
print("Model weights saved to pytorch_model.pth. The architecture is defined by FashionClassifier above.")